In [1]:
from pathlib import Path

import kagglehub
import pandas as pd

dataset_dir = kagglehub.dataset_download(
    "shivamb/real-or-fake-fake-jobposting-prediction"
)

csv_path = Path(dataset_dir) / "fake_job_postings.csv"
df = pd.read_csv(csv_path)

print("Dataset shape:", df.shape)
df[["title", "description", "fraudulent"]].head()

Using Colab cache for faster access to the 'real-or-fake-fake-jobposting-prediction' dataset.
Dataset shape: (17880, 18)


,title,description,fraudulent
0,Marketing Intern,"Food52, a fast-growing, James Beard Award-winn...",0
1,Customer Service - Cloud Video Production,Organised - Focused - Vibrant - Awesome!Do you...,0
2,Commissioning Machinery Assistant (CMA),"Our client, located in Houston, is actively se...",0
3,Account Executive - Washington DC,THE COMPANY: ESRI – Environmental Systems Rese...,0
4,Bill Review Manager,JOB TITLE: Itemization Review ManagerLOCATION:...,0


In [2]:
class_counts = df["fraudulent"].value_counts().sort_index()

class_summary = pd.DataFrame({
    "Count": class_counts,
    "Percentage": (class_counts / class_counts.sum() * 100).round(2)
})

class_summary = class_summary.rename(index={0: "Real", 1: "Fake"})
class_summary

,Count,Percentage
fraudulent,,
Real,17014,95.16
Fake,866,4.84


In [3]:
missing_counts = df.isna().sum().sort_values(ascending=False)

missing_summary = pd.DataFrame({
    "Missing count": missing_counts,
    "Missing percentage": (missing_counts / len(df) * 100).round(2)
})

display(missing_summary.head(8))

duplicate_count = df.drop(columns="job_id").duplicated().sum()
print("Duplicate rows excluding job_id:", duplicate_count)

,Missing count,Missing percentage
salary_range,15012,83.96
department,11547,64.58
required_education,8105,45.33
benefits,7212,40.34
required_experience,7050,39.43
function,6455,36.10
industry,4903,27.42
employment_type,3471,19.41


Duplicate rows excluding job_id: 281


In [4]:
comparison_columns = df.columns.drop("job_id")
clean_df = df.drop_duplicates(subset=comparison_columns).copy()

print("Rows before cleaning:", len(df))
print("Rows after removing duplicates:", len(clean_df))
print("Missing descriptions:", clean_df["description"].isna().sum())


Rows before cleaning: 17880
Rows after removing duplicates: 17599
Missing descriptions: 1


In [5]:
model_df = clean_df.dropna(subset=["description"])

X = model_df["description"]
y = model_df["fraudulent"]

print("Rows available for modeling:", len(model_df))
print("Input samples:", len(X))
print("Target labels:", len(y))

Rows available for modeling: 17598
Input samples: 17598
Target labels: 17598


In [6]:
from sklearn.model_selection import train_test_split

X_train, X_remaining, y_train, y_remaining = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=42
)

X_val, X_test, y_val, y_test = train_test_split(
    X_remaining,
    y_remaining,
    test_size=0.50,
    stratify=y_remaining,
    random_state=42
)

for name, labels in [
    ("Training", y_train),
    ("Validation", y_val),
    ("Test", y_test),
]:
    print(f"{name}: {len(labels)} samples, {labels.mean() * 100:.2f}% fake")


Training: 12318 samples, 4.85% fake
Validation: 2640 samples, 4.89% fake
Test: 2640 samples, 4.85% fake


In [7]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.pipeline import make_pipeline

baseline_model = make_pipeline(
    TfidfVectorizer(),
    LogisticRegression(max_iter=1000)
)

baseline_model.fit(X_train, y_train)
validation_predictions = baseline_model.predict(X_val)

print(classification_report(
    y_val,
    validation_predictions,
    labels=[0, 1],
    target_names=["Real", "Fake"],
    digits=3,
    zero_division=0
))

print("Confusion matrix:")
print(confusion_matrix(y_val, validation_predictions, labels=[0, 1]))

              precision    recall  f1-score   support

        Real      0.965     1.000     0.982      2511
        Fake      1.000     0.295     0.455       129

    accuracy                          0.966      2640
   macro avg      0.983     0.647     0.719      2640
weighted avg      0.967     0.966     0.956      2640

Confusion matrix:
[[2511    0]
 [  91   38]]


In [8]:
weighted_model = make_pipeline(
    TfidfVectorizer(),
    LogisticRegression(max_iter=1000, class_weight="balanced")
)

weighted_model.fit(X_train, y_train)
weighted_predictions = weighted_model.predict(X_val)

print(classification_report(
    y_val,
    weighted_predictions,
    labels=[0, 1],
    target_names=["Real", "Fake"],
    digits=3,
    zero_division=0
))

print("Confusion matrix:")
print(confusion_matrix(y_val, weighted_predictions, labels=[0, 1]))

              precision    recall  f1-score   support

        Real      0.987     0.970     0.979      2511
        Fake      0.563     0.760     0.647       129

    accuracy                          0.959      2640
   macro avg      0.775     0.865     0.813      2640
weighted avg      0.967     0.959     0.962      2640

Confusion matrix:
[[2435   76]
 [  31   98]]


In [9]:
from sklearn.naive_bayes import MultinomialNB

naive_bayes_model = make_pipeline(
    TfidfVectorizer(),
    MultinomialNB()
)

naive_bayes_model.fit(X_train, y_train)
naive_bayes_predictions = naive_bayes_model.predict(X_val)

print(classification_report(
    y_val,
    naive_bayes_predictions,
    labels=[0, 1],
    target_names=["Real", "Fake"],
    digits=3,
    zero_division=0
))

print("Confusion matrix:")
print(confusion_matrix(y_val, naive_bayes_predictions, labels=[0, 1]))

              precision    recall  f1-score   support

        Real      0.951     1.000     0.975      2511
        Fake      0.000     0.000     0.000       129

    accuracy                          0.951      2640
   macro avg      0.476     0.500     0.487      2640
weighted avg      0.905     0.951     0.927      2640

Confusion matrix:
[[2510    1]
 [ 129    0]]


In [10]:
from sklearn.svm import LinearSVC

svm_model = make_pipeline(
    TfidfVectorizer(),
    LinearSVC(random_state=42)
)

weighted_svm_model = make_pipeline(
    TfidfVectorizer(),
    LinearSVC(class_weight="balanced", random_state=42)
)

for name, model in [
    ("Linear SVM", svm_model),
    ("Weighted Linear SVM", weighted_svm_model)
]:
    model.fit(X_train, y_train)
    predictions = model.predict(X_val)

    print(f"\n{name}")
    print(classification_report(
        y_val,
        predictions,
        labels=[0, 1],
        target_names=["Real", "Fake"],
        digits=3,
        zero_division=0
    ))

    print("Confusion matrix:")
    print(confusion_matrix(y_val, predictions, labels=[0, 1]))


Linear SVM
              precision    recall  f1-score   support

        Real      0.977     1.000     0.988      2511
        Fake      0.986     0.550     0.706       129

    accuracy                          0.978      2640
   macro avg      0.982     0.775     0.847      2640
weighted avg      0.978     0.978     0.975      2640

Confusion matrix:
[[2510    1]
 [  58   71]]

Weighted Linear SVM
              precision    recall  f1-score   support

        Real      0.983     0.993     0.988      2511
        Fake      0.829     0.674     0.744       129

    accuracy                          0.977      2640
   macro avg      0.906     0.834     0.866      2640
weighted avg      0.976     0.977     0.976      2640

Confusion matrix:
[[2493   18]
 [  42   87]]


In [11]:
from sklearn.metrics import f1_score

validation_scores = weighted_svm_model.decision_function(X_val)

for threshold in [-0.50, -0.25, -0.10, 0.00, 0.10, 0.25, 0.50]:
    predictions = (validation_scores > threshold).astype(int)

    _, false_positives, false_negatives, true_positives = confusion_matrix(
        y_val, predictions, labels=[0, 1]
    ).ravel()

    fake_f1 = f1_score(y_val, predictions, pos_label=1)

    print(
        f"Threshold {threshold:+.2f} | Fake F1 {fake_f1:.3f} | "
        f"Detected {true_positives}/{len(y_val[y_val == 1])} | "
        f"Missed {false_negatives} | False alarms {false_positives}"
    )

Threshold -0.50 | Fake F1 0.575 | Detected 107/129 | Missed 22 | False alarms 136
Threshold -0.25 | Fake F1 0.701 | Detected 96/129 | Missed 33 | False alarms 49
Threshold -0.10 | Fake F1 0.730 | Detected 92/129 | Missed 37 | False alarms 31
Threshold +0.00 | Fake F1 0.744 | Detected 87/129 | Missed 42 | False alarms 18
Threshold +0.10 | Fake F1 0.749 | Detected 85/129 | Missed 44 | False alarms 13
Threshold +0.25 | Fake F1 0.744 | Detected 80/129 | Missed 49 | False alarms 6
Threshold +0.50 | Fake F1 0.706 | Detected 71/129 | Missed 58 | False alarms 1


In [12]:
from sklearn.metrics import roc_auc_score

selected_threshold = 0.10
test_scores = weighted_svm_model.decision_function(X_test)
test_predictions = (test_scores > selected_threshold).astype(int)

print(classification_report(
    y_test,
    test_predictions,
    labels=[0, 1],
    target_names=["Real", "Fake"],
    digits=3,
    zero_division=0
))

print("Confusion matrix:")
print(confusion_matrix(y_test, test_predictions, labels=[0, 1]))

print(f"ROC AUC: {roc_auc_score(y_test, test_scores):.3f}")

              precision    recall  f1-score   support

        Real      0.984     0.993     0.989      2512
        Fake      0.830     0.688     0.752       128

    accuracy                          0.978      2640
   macro avg      0.907     0.840     0.870      2640
weighted avg      0.977     0.978     0.977      2640

Confusion matrix:
[[2494   18]
 [  40   88]]
ROC AUC: 0.944


In [13]:
import joblib
import sklearn
from google.colab import files

model_path = "fake_job_model.joblib"

joblib.dump(
    {
        "pipeline": weighted_svm_model,
        "threshold": selected_threshold,
        "sklearn_version": sklearn.__version__,
    },
    model_path,
    compress=3,
)

print("Model saved:", model_path)
print("scikit-learn version:", sklearn.__version__)
files.download(model_path)

Model saved: fake_job_model.joblib
scikit-learn version: 1.6.1


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>